In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA = "sdn.csv"
TARGET = "label"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
df = pd.read_csv(DATA)
df = df.replace([np.inf, -np.inf], np.nan).drop_duplicates()
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

In [4]:
string_cols = df.drop(columns=[TARGET]).select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
df = df.drop(columns=string_cols)

for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [5]:
features = [c for c in df.columns if c != TARGET]
leakage = []
for c in features:
    x, y = df[c], df[TARGET]
    exact = x.notna().all() and np.array_equal(x.to_numpy(), y.to_numpy())
    g = pd.DataFrame({"x": x, "y": y}).dropna().groupby("x")["y"].nunique()
    perfect = len(g) and g.max() <= 1
    corr = abs(x.corr(y)) if x.nunique(dropna=True) > 1 else 0
    leakage.append((c, exact, perfect, corr))

In [6]:
leakage_cols = [c for c,e,p,corr in leakage if e or p]
print("Definite leakage columns:", leakage_cols)
df = df.drop(columns=leakage_cols)

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

Definite leakage columns: []


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
X_train = scaler.fit_transform(imputer.fit_transform(X_train)).astype("float32")
X_test = scaler.transform(imputer.transform(X_test)).astype("float32")

classes, counts = np.unique(y_train, return_counts=True)
minority = classes[np.argmin(counts)]
n_new = int(counts.max() - counts.min())
X_min = X_train[y_train.to_numpy() == minority]

In [8]:
class Generator(nn.Module):
    def __init__(self, zdim, nf):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(zdim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, nf)
        )
    def forward(self, z): return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, nf):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nf, 64), nn.LeakyReLU(.2),
            nn.Linear(64, 32), nn.LeakyReLU(.2),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

In [9]:
zdim = 32
nf = X_train.shape[1]
G = Generator(zdim, nf).to(device)
D = Discriminator(nf).to(device)

loader = DataLoader(
    TensorDataset(torch.tensor(X_min)),
    batch_size=min(1024, len(X_min)),
    shuffle=True,
    pin_memory=(device.type == "cuda")
)

bce = nn.BCELoss()
optG = torch.optim.Adam(G.parameters(), lr=2e-4)
optD = torch.optim.Adam(D.parameters(), lr=2e-4)

In [10]:
GAN_EPOCHS = 15
for epoch in range(GAN_EPOCHS):
    for (real,) in loader:
        real = real.to(device, non_blocking=True)
        n = len(real)

        z = torch.randn(n, zdim, device=device)
        fake = G(z)

        optD.zero_grad(set_to_none=True)
        dloss = (
            bce(D(real), torch.ones(n,1,device=device)) +
            bce(D(fake.detach()), torch.zeros(n,1,device=device))
        )
        dloss.backward()
        optD.step()

        z = torch.randn(n, zdim, device=device)
        optG.zero_grad(set_to_none=True)
        gloss = bce(D(G(z)), torch.ones(n,1,device=device))
        gloss.backward()
        optG.step()


In [11]:
G.eval()
synthetic = []
with torch.no_grad():
    for start in range(0, n_new, 4096):
        n = min(4096, n_new-start)
        synthetic.append(
            G(torch.randn(n, zdim, device=device)).cpu().numpy()
        )

X_syn = np.vstack(synthetic).astype("float32")
y_syn = np.full(len(X_syn), minority, dtype="int64")

X_aug = np.vstack([X_train, X_syn])
y_aug = np.concatenate([y_train.to_numpy(), y_syn])

perm = np.random.permutation(len(y_aug))
X_aug, y_aug = X_aug[perm], y_aug[perm]

In [12]:
class CNN1D(nn.Module):
    def __init__(self, nf, nc):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 96, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(96, 32), nn.ReLU(), nn.Dropout(.25),
            nn.Linear(32, nc)
        )

    def forward(self, x):
        return self.head(self.features(x))

model = CNN1D(nf, len(classes)).to(device)
train_ds = TensorDataset(
    torch.tensor(X_aug[:,None,:]),
    torch.tensor(y_aug)
)
train_loader = DataLoader(
    train_ds, batch_size=1024, shuffle=True,
    pin_memory=(device.type == "cuda")
)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

CNN_EPOCHS = 100
for epoch in range(CNN_EPOCHS):
    model.train()
    total = 0
    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(yb)
    print(f"Epoch {epoch+1}/{CNN_EPOCHS} loss={total/len(train_ds):.4f}")

Epoch 1/100 loss=0.3872
Epoch 2/100 loss=0.1730
Epoch 3/100 loss=0.1340
Epoch 4/100 loss=0.1119
Epoch 5/100 loss=0.1002
Epoch 6/100 loss=0.0868
Epoch 7/100 loss=0.0809
Epoch 8/100 loss=0.0707
Epoch 9/100 loss=0.0719
Epoch 10/100 loss=0.0642
Epoch 11/100 loss=0.0589
Epoch 12/100 loss=0.0566
Epoch 13/100 loss=0.0496
Epoch 14/100 loss=0.0471
Epoch 15/100 loss=0.0432
Epoch 16/100 loss=0.0369
Epoch 17/100 loss=0.0382
Epoch 18/100 loss=0.0360
Epoch 19/100 loss=0.0342
Epoch 20/100 loss=0.0309
Epoch 21/100 loss=0.0296
Epoch 22/100 loss=0.0342
Epoch 23/100 loss=0.0301
Epoch 24/100 loss=0.0271
Epoch 25/100 loss=0.0303
Epoch 26/100 loss=0.0234
Epoch 27/100 loss=0.0225
Epoch 28/100 loss=0.0224
Epoch 29/100 loss=0.0212
Epoch 30/100 loss=0.0200
Epoch 31/100 loss=0.0206
Epoch 32/100 loss=0.0205
Epoch 33/100 loss=0.0170
Epoch 34/100 loss=0.0298
Epoch 35/100 loss=0.0172
Epoch 36/100 loss=0.0232
Epoch 37/100 loss=0.0172
Epoch 38/100 loss=0.0185
Epoch 39/100 loss=0.0166
Epoch 40/100 loss=0.0151
Epoch 41/

In [13]:
model.eval()
with torch.no_grad():
    test_tensor = torch.tensor(X_test[:,None,:]).to(device)
    pred = model(test_tensor).argmax(1).cpu().numpy()

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, pred, digits=4))
print("CONFUSION MATRIX")
print(confusion_matrix(y_test, pred))


CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.9958    0.9987    0.9972     12250
           1     0.9979    0.9932    0.9955      7601

    accuracy                         0.9966     19851
   macro avg     0.9968    0.9959    0.9964     19851
weighted avg     0.9966    0.9966    0.9966     19851

CONFUSION MATRIX
[[12234    16]
 [   52  7549]]
